# Management Agent

> Agent that manages multiple tool using agents

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp agents.manager_agent

In [ ]:
#| hide
from nbdev.showdoc import *
from rich.pretty import pprint

In [ ]:
#| export
import datetime
from dataclasses import dataclass
from typing import Literal
from httpx import AsyncClient, ConnectError, ConnectTimeout, RemoteProtocolError
import requests

from pydantic import BaseModel, Field
from rich.prompt import Prompt

from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.messages import ModelMessage
from pydantic_ai.usage import Usage, UsageLimits
from pydantic_ai.tools import Tool

from agent_chatbot.agents.orchestration_agent import build_orchestration_agent, _OrchestrationPlan

In [ ]:
#| export
class _ManagementResponse(BaseModel):
    """Response from the manager"""
    final_result: str = Field(..., description="The final result of the request structured using markdown and latex")
    final_goal_completed: bool = Field(..., description="Whether the final goal was completed")
    final_goal: str = Field(..., description="The final goal of the request")
    evedence: list[str] = Field(..., description="The evidence for the final result")
    plan: list[str] = Field(..., description="The plan to achieve the final goal")
    plan_completed: bool = Field(..., description="Whether the plan was completed")

In [ ]:
from duckduckgo_search import DDGS

In [ ]:
@dataclass
class SearchSolution:
    sub_goal: str
    search_results: list[str]
    result_summary: str

@dataclass
class SearchDeps:
    search_client: DDGS

In [ ]:
@dataclass
class Deps:
    orchestration_agent: Agent[None, _OrchestrationPlan]
    search_agent: Agent[SearchDeps, SearchSolution]
    available_tools: dict[str, Tool]

In [ ]:
#| eval: false
management_agent = Agent[Deps, _ManagementResponse](
    "openai:gpt-4o-mini",
    result_type=_ManagementResponse,
    deps_type=Deps,
    system_prompt=(
        "You are a manager of a set of expert ai agents. You need to manage the agents to achieve a goal as efficiently as possible. ",
        "You may need to ask the agents for information, or to perform tasks. You must use the `orchestration_agent_tool` to plan the tasks.",
        "Use the `orchestration_agent_tool` to plan the tasks into smaller sub-goals, and then ask the agents to perform the sub-goals.",
        "You must try to complete all `sub_goals` as suggested by the `plan` from the `orchestration_agent_tool` to achieve the final goal. ",
        "You can use the `search_tool` to search for information to help you achieve each sub-goal. ",
        "Make sure to reference the `evidence` from the agents to help you make decisions. ",
    )
)

In [ ]:
#| eval: false
@management_agent.tool
async def orchestration_agent_tool(ctx: RunContext, goal: str) -> _OrchestrationPlan:
    """This tool can be used to formulate a plan of action to achieve a goal"""
    
    return await ctx.deps.orchestration_agent.run(goal)

In [ ]:
#| eval: false
search_agent = Agent[SearchDeps, SearchSolution|None](
    "google-gla:gemini-1.5-flash",
    result_type=SearchSolution,
    deps_type=SearchDeps,
    system_prompt=(
        "You are an expert at searching the web for information. "
        "You can quickly find information on any topic and provide a summary of the results. "
        "You have been asked to help a user find information on with a specific sub-goal in mind. "
        "You have been provided with a prompt to search for please provide a summary of the results. "
    )
)

In [ ]:
#| eval: false
@search_agent.tool
def search_tool(ctx: RunContext[SearchDeps], query: str) -> str:
    """Use search client to find information on a query"""
    return ctx.deps.search_client.text(query, region='en-us', max_results=10)

In [ ]:
#| eval: false
@management_agent.tool
async def search_tool(ctx: RunContext, query: str) -> SearchSolution:
    """Use search agent to find information on a query"""
    search_results = await ctx.deps.search_agent.run(query, deps=SearchDeps(search_client=DDGS()))
    return search_results

In [ ]:
#| eval: false
orchestration_agent = build_orchestration_agent()

In [ ]:
#@orchestration_agent.tool_plain
#def list_available_tools() -> list[str]:
#    """List the available tools"""
#    return list(management_agent._function_tools.keys())

In [ ]:
#| eval: false
result = management_agent.run(
    "Can you explain some issues that may arise when using OLS regression for trending timeseries seasonal data? And how to address them? Please provide an example case study.",
    deps=Deps(
        orchestration_agent=orchestration_agent,
        search_agent=search_agent,
        available_tools=management_agent._function_tools
    ))


In [ ]:
#| eval: false
message = await result

In [ ]:
#| eval: false
from IPython.display import display, Markdown

In [ ]:
#| eval: false
#| echo: false
Markdown(message.data.final_result)

### Issues with Using OLS Regression for Trending Time Series Seasonal Data

1. **Limitations of OLS Regression**
   - OLS regression assumes that the data is stationary, which means that the statistical properties such as mean and variance do not change over time. Trending time series data often violate this assumption.
   - The presence of trends can lead to spurious results, meaning that the model may indicate a strong correlation between variables when there is none due to the trend itself driving the relationship.
   - OLS is sensitive to autocorrelation in residuals, which is common in time series data. This can lead to inefficient estimates and biased coefficient values.  

   **Evidence:**  OLS results on trending time series can appear robust but are fundamentally flawed, as demonstrated in various studies. 
      - **Reference:**  [Montana State University](https://www.montana.edu/cstoddard/562/Autocorrelation.pdf)
   
2. **Seasonality Issues in Time Series Data**
   - Seasonality leads to patterns that repeat at regular intervals and can introduce autocorrelation in the residuals of an OLS model.
   - This autocorrelation violates the OLS assumption regarding the independence of errors, resulting in biased estimates of coefficients and unreliable standard errors. 
   - If seasonality is ignored, it can result in incorrect conclusions and forecasts. 

   **Evidence:** Including seasonal patterns in time series data without proper modeling ends up with biased coefficients.
      - **Reference:** [Cumulative Approach on Seasonal Analysis](https://prof-rossetti.github.io/predictive-modeling-python-book/notes/time-series-forecasting/seasonality.html) 

3. **Methods to Address OLS Issues**
   - **Transformations:** Techniques like differencing (removing trends) or logarithmic transformations (to stabilize variance) can be employed.
   - **Include Dummy Variables:** Adding seasonal dummy variables helps in accounting for seasonality directly in the regression model.
   - **Alternative Modeling Techniques:** Models like ARIMA are specifically designed to handle autocorrelation and trends in time series data, allowing for better predictions and hypothesis testing.
     - **Note:** AIC and BIC are useful criteria for selecting the best-fitting ARIMA model. 

   **Evidence:** Various studies have shown that transforming data or using ARIMA leads to better results in terms of forecast accuracy. 
      - **Reference:** [SpringerLink on Time Series Forecasting](https://link.springer.com/chapter/10.1007/978-3-031-28113-6_6)

### Example Case Study:
- **Hypothetical Dataset:** Consider a dataset capturing monthly sales data over several years, which displays both a trend and seasonal effects (e.g., higher sales during holidays).
- **Problem:** Applying OLS regression without accounting for trend and seasonality might initially yield a high R-squared value, suggesting a good fit. However, failure to include seasonal dummies and differences would understate the actual relationships.
- **Solution:** By differencing the data and adding seasonal dummy variables, the regression model would not only fit the data better but also provide unbiased estimates of the relationship between marketing efforts and sales. Models like ARIMA would be tested for further accuracy improvements.

### Conclusion:
While OLS can provide a preliminary analysis of time series data, its limitations necessitate careful consideration of trends, seasonality, and alternative models for reliable outcomes.

In [ ]:
#| eval: false
pprint(message.all_messages()[2].parts[0].content.data)

_OrchestrationPlan(
│   main_goal='Explain issues with using OLS regression for trending timeseries seasonal data and how to address them, providing an example case study.',
│   sub_goals=[
│   │   'Identify the limitations of OLS regression in handling trending timeseries data.',
│   │   'Discuss issues related to seasonality in time series data when using OLS regression.',
│   │   'Outline methods to address these issues, including alternative modeling techniques.',
│   │   'Provide a detailed case study example illustrating these concepts and solutions.'
│   ],
│   plan=[
│   │   'Identify the specific problems with OLS regression for trending timeseries, such as non-stationarity, autocorrelation, and the impact of seasonality.',
│   │   'Explain how seasonality can create bias in regression coefficients and affect predictions.',
│   │   'Discuss methodologies to address these issues, including transformations and differencing, and alternative methods such as ARIMA or structural time series models.',
│   │   'Create a case study example using a hypothetical dataset that demonstrates the application of OLS regression to trending seasonal data, highlighting the issues faced.',
│   │   'Present solutions applied in the case study, comparing the results of OLS regression vs. a more suitable model for the data.'
│   ]
)

In [ ]:
#| eval: false
for part in message.all_messages()[4].parts:
    try:
        pprint(part.content.data)
    except KeyError:
        pass

SearchSolution(
│   sub_goal='limitations of OLS regression in trending timeseries data',
│   search_results=[
│   │   'https://www.montana.edu/cstoddard/562/Autocorrelation.pdf',
│   │   'https://stats.stackexchange.com/questions/94723/using-non-stationary-time-series-data-in-ols-regression',
│   │   'https://fenix.iseg.ulisboa.pt/downloadFile/281608120779331/Ch_09.pdf',
│   │   'https://orbi.uliege.be/bitstream/2268/266529/6/Lecture+Notes+Ch+10-11+-+RegressionWithTimeSeriesData.pdf',
│   │   'https://web.ics.purdue.edu/~bvankamm/Files/360+Notes/09+-+Regression+with+Time+Series+Data.pdf',
│   │   'https://is.muni.cz/el/econ/jaro2022/MPE_ECNM/um/ECNM_Lecture_11_Spring_2022.pdf',
│   │   'https://fenix.iseg.ulisboa.pt/downloadFile/844558074123184/Lecture+10.pdf',
│   │   'https://www.researchgate.net/post/Do-I-need-to-stationnarize-time-series-in-an-OLS-regression-with-time-trend',
│   │   'https://docslib.org/doc/12699009/issues-using-ols-with-time-series-data',
│   │   'https://bigdataconference.eu/wp-content/uploads/2018/12/Working-with-Outliers-and-Time-Series-Shocks-by-Michael-Grogan-min.pdf'
│   ],
│   result_summary="Ordinary Least Squares (OLS) regression, while widely used, has limitations when applied to trending time series data.  OLS assumes stationarity, meaning the statistical properties of the data don't change over time. Trending data violates this assumption.  This can lead to spurious regressions, where high R-squared values might suggest a strong relationship, but the results are meaningless because the trends are driving the correlation, not a genuine underlying relationship.  Furthermore, OLS is sensitive to autocorrelation (correlation between successive error terms), common in time series.  This can lead to inefficient and biased estimates of coefficients.  To address these issues, transformations like differencing or detrending are often employed before applying OLS, or other time series specific methods such as ARIMA modeling should be considered."
)

SearchSolution(
│   sub_goal='impact of seasonality in time series data when using OLS regression',
│   search_results=[
│   │   'Using OLS regression on time series data with seasonality can lead to biased and inefficient estimates if the seasonality is not properly accounted for.  Seasonality violates the OLS assumption of independent errors.  Methods for handling seasonality include adding seasonal dummy variables to the regression model, or using time series models designed to handle seasonality such as ARIMA models.',
│   │   'Ignoring seasonality in OLS regression with time series data results in biased coefficient estimates and inaccurate standard errors, leading to unreliable hypothesis tests and forecasts.  Seasonal dummy variables or other methods are necessary to adjust for this violation of the OLS assumptions.',
│   │   'The impact of seasonality in OLS regression on time series data is significant.  Seasonality introduces autocorrelation in the residuals, violating the assumption of independent errors.  This leads to inefficient and biased estimates of regression parameters.  Various techniques exist to mitigate this problem, including the use of seasonal dummy variables and other specialized time series models.',
│   │   'Seasonal patterns in time series data can cause autocorrelation, violating the independence of errors assumption in OLS regression. This results in inefficient and potentially biased coefficient estimates and invalid standard errors.  Solutions involve incorporating seasonal dummy variables or utilizing more sophisticated time series methods such as ARIMA models that explicitly model seasonality.',
│   │   'OLS regression on time series data with seasonal components requires special consideration.  The presence of seasonality violates the independence of error assumption, potentially leading to biased coefficient estimates and incorrect standard errors.  Proper handling of seasonality is crucial for accurate analysis and forecasting. Various methods, such as including seasonal dummy variables, differencing, or employing ARIMA models, can address this issue.'
│   ],
│   result_summary='Seasonality in time series data, if not properly accounted for, can lead to biased and inefficient OLS regression results.  Standard OLS assumes independence of errors, a condition often violated by seasonal patterns.  Methods to address this include incorporating seasonal dummy variables (representing each season as a separate variable) or using time series models explicitly designed to handle seasonality, such as ARIMA models.  Failure to account for seasonality can lead to inaccurate forecasts and misleading interpretations of regression coefficients.'
)

SearchSolution(
│   sub_goal='methods to address OLS regression issues in time series data including ARIMA',
│   search_results=[
│   │   'https://stats.stackexchange.com/questions/379520/selecting-between-ols-regression-and-arima-for-time-series-why-aic-or-bic-for-a',
│   │   'https://online.stat.psu.edu/stat510/lesson/8/8.1',
│   │   'https://www.montana.edu/cstoddard/562/Autocorrelation.pdf',
│   │   'https://online.stat.psu.edu/stat510/book/export/html/669',
│   │   'https://www.montana.edu/cstoddard/562/Autocorrelation.pdf',
│   │   'https://fenix.iseg.ulisboa.pt/downloadFile/281608120779331/Ch_09.pdf',
│   │   'https://link.springer.com/chapter/10.1007/978-3-031-28113-6_6',
│   │   'https://methods.sagepub.com/book/mono/introduction-to-time-series-analysis/chpt/static-time-series-models-ordinary-least-squares-estimation',
│   │   'https://is.muni.cz/el/econ/jaro2022/MPE_ECNM/um/ECNM_Lecture_11_Spring_2022.pdf',
│   │   'https://www.researchgate.net/publication/366297281_Comparison_Between_ARIMA_Model_and_OLS_Model_Based_on_the_Economic_Representation/fulltext/639c36dab260ef307fd7973b/Comparison-Between-ARIMA-Model-and-OLS-Model-Based-on-the-Economic-Representation.pdf'
│   ],
│   result_summary='Several methods exist to address OLS regression issues in time series data, including those involving autocorrelation and non-stationarity.  Common approaches involve transforming the data (e.g., differencing, logarithmic transformation), using robust standard errors (e.g., Newey-West), or incorporating ARIMA models to account for temporal dependence in the residuals.  ARIMA models can directly model the time series structure, potentially handling issues that simple OLS regression might miss. The choice of method depends on the specific characteristics of the data and the nature of the autocorrelation or other issues present.  AIC and BIC can help in selecting the best ARIMA model.'
)

In [ ]:
#| eval: false
pprint(message.data)

_ManagementResponse(
│   final_result='### Issues with Using OLS Regression for Trending Time Series Seasonal Data\n\n1. **Limitations of OLS Regression**\n   - OLS regression assumes that the data is stationary, which means that the statistical properties such as mean and variance do not change over time. Trending time series data often violate this assumption.\n   - The presence of trends can lead to spurious results, meaning that the model may indicate a strong correlation between variables when there is none due to the trend itself driving the relationship.\n   - OLS is sensitive to autocorrelation in residuals, which is common in time series data. This can lead to inefficient estimates and biased coefficient values.  \n\n   **Evidence:**  OLS results on trending time series can appear robust but are fundamentally flawed, as demonstrated in various studies. \n      - **Reference:**  [Montana State University](https://www.montana.edu/cstoddard/562/Autocorrelation.pdf)\n   \n2. **Seasonality Issues in Time Series Data**\n   - Seasonality leads to patterns that repeat at regular intervals and can introduce autocorrelation in the residuals of an OLS model.\n   - This autocorrelation violates the OLS assumption regarding the independence of errors, resulting in biased estimates of coefficients and unreliable standard errors. \n   - If seasonality is ignored, it can result in incorrect conclusions and forecasts. \n\n   **Evidence:** Including seasonal patterns in time series data without proper modeling ends up with biased coefficients.\n      - **Reference:** [Cumulative Approach on Seasonal Analysis](https://prof-rossetti.github.io/predictive-modeling-python-book/notes/time-series-forecasting/seasonality.html) \n\n3. **Methods to Address OLS Issues**\n   - **Transformations:** Techniques like differencing (removing trends) or logarithmic transformations (to stabilize variance) can be employed.\n   - **Include Dummy Variables:** Adding seasonal dummy variables helps in accounting for seasonality directly in the regression model.\n   - **Alternative Modeling Techniques:** Models like ARIMA are specifically designed to handle autocorrelation and trends in time series data, allowing for better predictions and hypothesis testing.\n     - **Note:** AIC and BIC are useful criteria for selecting the best-fitting ARIMA model. \n\n   **Evidence:** Various studies have shown that transforming data or using ARIMA leads to better results in terms of forecast accuracy. \n      - **Reference:** [SpringerLink on Time Series Forecasting](https://link.springer.com/chapter/10.1007/978-3-031-28113-6_6)\n\n### Example Case Study:\n- **Hypothetical Dataset:** Consider a dataset capturing monthly sales data over several years, which displays both a trend and seasonal effects (e.g., higher sales during holidays).\n- **Problem:** Applying OLS regression without accounting for trend and seasonality might initially yield a high R-squared value, suggesting a good fit. However, failure to include seasonal dummies and differences would understate the actual relationships.\n- **Solution:** By differencing the data and adding seasonal dummy variables, the regression model would not only fit the data better but also provide unbiased estimates of the relationship between marketing efforts and sales. Models like ARIMA would be tested for further accuracy improvements.\n\n### Conclusion:\nWhile OLS can provide a preliminary analysis of time series data, its limitations necessitate careful consideration of trends, seasonality, and alternative models for reliable outcomes.',
│   final_goal_completed=True,
│   final_goal='Explain issues that may arise when using OLS regression for trending timeseries seasonal data, and how to address them, providing an example case study.',
│   evedence=[
│   │   'https://www.montana.edu/cstoddard/562/Autocorrelation.pdf',
│   │   'https://prof-rossetti.github.io/predictive-modeling-python-book/notes/time-series-forecasting/seasona

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export();